In [1]:
!pip install torch torchvision pandas tqdm scikit-learn matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 2.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 79.3 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 69.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 KB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 79.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 KB 58.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 KB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 82.0 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 7.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.6 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

In [1]:
import torch, torchvision
print("torch:", torch.__version__, torch.__file__)
print("torchvision:", torchvision.__version__, torchvision.__file__)


torch: 2.10.0+cu128 /home/na1488tr-s/.local/lib/python3.10/site-packages/torch/__init__.py
torchvision: 0.25.0+cu128 /home/na1488tr-s/.local/lib/python3.10/site-packages/torchvision/__init__.py


In [1]:
import os 
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.transforms as T
from tqdm import tqdm

from Data_Utility.lookup_size import lookup_size_from_excel
from Data_Utility.dataset import PollenFolderWithSizeDataset

from models.basemodel import CNNWithSizeMLP

print("All libraries imported successfully!")

All libraries imported successfully!


## Preparing data

In [2]:
train_dir = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Sorted_224_sizeTrain'
test_dir = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Sorted_224_sizeTest'
test_excel_path = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/SizeTest/size_data_preds.xlsx'

test_size_lookup = lookup_size_from_excel(test_excel_path)

classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
class_to_idx = {c: i for i, c in enumerate(classes)}

test_tf = T.Compose([T.ToTensor()]) # converts PIL → Tensor

#train_dataset = PollenFolderWithSizeDataset(img_dir=train_dir, class_to_idx=class_to_idx, size_lookup=size_lookup)
test_dataset = PollenFolderWithSizeDataset(img_dir=test_dir, class_to_idx=class_to_idx, size_lookup=test_size_lookup, transform=test_tf)




## Base Model (Erik's)

In [12]:
#train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNNWithSizeMLP(num_classes=len(classes)).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)



In [17]:
def train_epoch(loader):
    model.train()
    total_loss = 0
    for imgs, sizes, labels in tqdm(loader):
        imgs = imgs.to(device)
        sizes = sizes.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs, sizes)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(loader):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for imgs, sizes, labels in loader:
            imgs = imgs.to(device)
            sizes = sizes.to(device)
            labels = labels.to(device)

            outputs = model(imgs, sizes)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [ ]:
epochs_num = 1000

for epoch in range(1, epochs_num + 1):
    train_loss = train_epoch(test_loader)
    val_acc = eval_epoch(test_loader)
    
    print(f"Epoch {epoch}: loss {train_loss:.4f}, val_acc {val_acc:.4f}")

100%|██████████| 61/61 [00:05<00:00, 11.77it/s]


Epoch 1: loss 0.0576, val_acc 0.9906


100%|██████████| 61/61 [00:05<00:00, 11.91it/s]


Epoch 2: loss 0.1461, val_acc 0.6207


100%|██████████| 61/61 [00:05<00:00, 11.99it/s]


Epoch 3: loss 0.1243, val_acc 0.9906


100%|██████████| 61/61 [00:05<00:00, 11.88it/s]


Epoch 4: loss 0.1044, val_acc 0.8855


 92%|█████████▏| 56/61 [00:04<00:00, 11.78it/s]

In [16]:
from collections import Counter

all_labels = [test_dataset[i][2].item() for i in range(len(test_dataset))]
counts = Counter(all_labels)

# show counts with class names
idx_to_class = {v: k for k, v in class_to_idx.items()}
for k in sorted(counts):
    print(k, idx_to_class[k], counts[k])


0 Bellis perennis 142
1 Brassica napus 200
2 Capsella bursa-pastoris 183
3 Cichorium intybus 130
4 Crepis capillaris 200
5 Hieracium umbellatum 200
6 Hypochaeris radicata 200
7 Sonchus arvensis 334
8 Tragopogon pratensis 166
9 Tussilago farfara 167
